In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Load the raw data
df = pd.read_csv('wokwi_simulation_files/vibration_data.csv', names=['timestamp', 'ax', 'ay', 'az', 'label'])

# 2. Define processing parameters
WINDOW_SIZE = 20  # 20 samples = ~1 second of data at 20Hz
features = []
labels = []

# 3. Slide the window across the data
for i in range(0, len(df) - WINDOW_SIZE, WINDOW_SIZE):
    window = df.iloc[i : i + WINDOW_SIZE]
    
    # Extract features for X, Y, and Z axes
    feature_row = [
        window['ax'].abs().mean(), window['ax'].std(), np.sqrt(np.mean(window['ax']**2)),
        window['ay'].abs().mean(), window['ay'].std(), np.sqrt(np.mean(window['ay']**2)),
        window['az'].abs().mean(), window['az'].std(), np.sqrt(np.mean(window['az']**2))
    ]
    
    # If more than half the window is faulty (1), label the whole window as faulty
    final_label = 1 if window['label'].sum() > (WINDOW_SIZE / 2) else 0
    
    features.append(feature_row)
    labels.append(final_label)

X = np.array(features)
y = np.array(labels)
print(f"📦 Feature matrix shape: {X.shape} (Windows, Features)")

#Random Forest
# 1. Split into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Initialize and train the classifier
model = RandomForestClassifier(n_estimators=10, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# 3. Evaluate performance
y_pred = model.predict(X_test)
print("\n📝 --- MODEL PERFORMANCE REPORT ---")
print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Faulty (1)']))

print("📊 --- CONFUSION MATRIX ---")
print(confusion_matrix(y_test, y_pred))




📦 Feature matrix shape: (127, 9) (Windows, Features)

📝 --- MODEL PERFORMANCE REPORT ---
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00        11
  Faulty (1)       1.00      1.00      1.00        15

    accuracy                           1.00        26
   macro avg       1.00      1.00      1.00        26
weighted avg       1.00      1.00      1.00        26

📊 --- CONFUSION MATRIX ---
[[11  0]
 [ 0 15]]
